Business Understanding 

Hotel Management team want to improve operations, revenue and customer satisfaction using historical booking data by ... 

- Reducing revenue loss due to cancellations
  - Achieved by predicting cancellations and intervene with deals    and incentives (classification)

- Forecast average daily rates (ADR) and total revenue
  - Predict ADR for a booking to use dynamic pricing (regression)
  
- Segement customers to improve marketing and loyalty programs
  - Identify types of stays ("Last-minute business","early leisure", "Agent-booked") and then cluster them into low/medium/high revenue buckets for specific marketing strategies
  -Identify customer segements for tageted marketing campaigns




The framework used to solve this business problem is the CRSIP DM Framework. The Crisp-DM framework is used to structure data science projects and stands for Cross-industry standard process for data mining.

The process is as follows:
Business understanding > data understanding > data preparation > modelling evaluation > deployment 

This notebook will cover data understanding and data preparation. Business understanding has been explaineed in the cell above.

--------------------------------------------------------------------------------------------------------------------------------

Data Understanding 

Reading the hotel data and checking for the following:

- Column names
- Data Types 
- Data Shape
- Missing values
- Nulls 

Inspecting the data:

In [45]:
import pandas as pd
data = pd.read_csv('hotel_bookings.csv')
data.head()


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [46]:
# Checking for column names and data types
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

In [47]:
# Checking for number of rows and columns
data.shape

(119390, 32)

In [48]:
data.isnull().sum() 


hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

Data Prep/processing

I will need to drop the company column as most of the values are null. I will also need to fill the null values in the agent and country column with 'not listed'.

In [49]:
data = data.drop(columns=['company'])
data['agent'] = data['agent'].fillna('Not Listed')
data['country'] = data['country'].fillna('Not Listed')
data.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,3,No Deposit,Not Listed,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,4,No Deposit,Not Listed,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,No Deposit,Not Listed,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,No Deposit,304.0,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,No Deposit,240.0,0,Transient,98.0,0,1,Check-Out,2015-07-03


Feature engineering and encoding existing and new columns to facitate analysis. This features may be altered during the process of modelling.

Modelling will be covered in subsequent cells.

In [52]:
data['total_nights'] = data['stays_in_weekend_nights'] + data['stays_in_week_nights']
data['total_guests'] = data['adults'] + data['children'] + data['babies']
data['total_revenue'] = data['adr'] * data['total_nights']
data['Weekend/weekday ratio'] = data['stays_in_weekend_nights'] / (data['stays_in_week_nights'])    

In [50]:
data['lead_time_bins'] = pd.cut(data['lead_time'], 
                               bins=[0,7,30,90,180,365,2000], 
                               labels=['<1 week (Short Term)', '1-4 weeks (Short Term)', '1-3 months (Mid Term)', '3-6 months (Mid Term)', '6-12 months (Long Term)', '>1 year (Long Term)'])
data["is_family"] = ((data['children'] > 0)| (data['babies'] > 0)).astype(int)

data['lead_time_bins'].head()

0    6-12 months (Long Term)
1        >1 year (Long Term)
2       <1 week (Short Term)
3     1-4 weeks (Short Term)
4     1-4 weeks (Short Term)
Name: lead_time_bins, dtype: category
Categories (6, object): ['<1 week (Short Term)' < '1-4 weeks (Short Term)' < '1-3 months (Mid Term)' < '3-6 months (Mid Term)' < '6-12 months (Long Term)' < '>1 year (Long Term)']

In [53]:
import pandas as pd

# Dictionary mapping month → season
season_map = {
    "December": "Winter", "January": "Winter", "February": "Winter",
    "March": "Spring", "April": "Spring", "May": "Spring",
    "June": "Summer", "July": "Summer", "August": "Summer",
    "September": "Autumn", "October": "Autumn", "November": "Autumn"
}

# Create the season column
data["season"] = data["arrival_date_month"].map(season_map)

data["season"].head()

0    Summer
1    Summer
2    Summer
3    Summer
4    Summer
Name: season, dtype: object

----------------------------

Challenges faced in the data preparation phase:

A significiant amount null values were found in the data and required feature engineering (such as encoding and creating new features) to gain useful insights. This was a manual but relatively fast process. The step that required the most iterations was categorising and mapping 'lead time bins' and 'seasons', as this was a step that I had not done previously. Therefore, it was a process of trial and error. In order to overcome this I needed to research online articles and consult an ai tool to understand how to map columns. These columns were then used to model to help produce outputs.